# Microgrid RL Lab - Law 82-21

## Learning goals
In this lab, you train a reinforcement learning agent to operate a simplified microgrid with a battery.

By the end, you should be able to explain how three design choices affect performance:
- the observation space seen by the agent,
- the reward shaping used during training,
- the PPO hyperparameters used to learn the policy.

## Environment overview
This notebook uses `MicrogridLawEnv`, a deterministic teaching environment inspired by Moroccan Law 82-21.

### Time scale
- One episode lasts **168 steps**, which corresponds to **7 days x 24 hours**.
- Each step represents one hour.

### Physical system
- The microgrid has solar generation, a local electrical demand, and one battery.
- Battery state of charge is capped at **100 units**.
- The maximum charge or discharge rate is **20 units per hour**.
- The battery starts each episode at **20 units**.

### Energy signals
- **Solar production** is deterministic and follows a smooth daylight curve between 06:00 and 18:00.
- **Demand** is deterministic and includes morning, midday, and evening peaks.
- **Tariff** is also deterministic: it is higher during peak evening hours and lower otherwise.

### Actions
The agent chooses one action at every hour:
- `0`: `IDLE` -> keep the battery unchanged,
- `1`: `CHARGE` -> store available energy in the battery,
- `2`: `DISCHARGE` -> use battery energy to cover demand or export surplus.

### Operational rule
- The environment enforces a **daily export cap of 20% of the day's generated solar energy**.
- This means the agent cannot simply sell unlimited energy to the grid.

### Observation space
You choose which features are exposed to the agent from this list:
- `soc`: normalized battery state of charge,
- `solar`: current solar generation,
- `demand`: current demand,
- `price`: current electricity tariff,
- `hour`: hour of day.

The order of `selected_features` matters because it defines the observation vector seen by the policy network.

### Reward structure
The environment always computes three interpretable reward terms:
- `profit`: money earned from selling energy minus money spent buying from the grid,
- `battery_use`: penalty proportional to battery activity,
- `grid_penalty`: penalty proportional to grid purchases.

Your final reward is a weighted sum of those terms, so reward shaping can encourage different behaviors.

### Why the IDLE baseline matters
Training an RL agent is only useful if it beats a simple reference strategy.
In this notebook we compare PPO against an `IDLE` baseline that always selects action `0`.
This gives a concrete estimate of the extra profit generated by learning.


In [ ]:
# =========================
# CONFIGURATION CELL (EDIT ONLY THIS)
# =========================

student_id = "student_01_ahmed"

# Observation space
selected_features = ["soc", "solar", "demand", "price", "hour"]

# Reward shaping
reward_weights = {
    "profit": 1,
    "battery_use": -1,
    "grid_penalty": -1,
}

# PPO hyperparameters
total_timesteps = 50_000
learning_rate = 3e-4
gamma = 0.99
entropy_coef = 0.01

# Reproducibility
seed = 42


In [ ]:
import json
import zipfile
from pathlib import Path

import gymnasium
import matplotlib.pyplot as plt
import numpy as np
import stable_baselines3
from stable_baselines3 import PPO

from env.microgrid_env import MicrogridLawEnv
from env.models import PANEL_COUNT, PANEL_RATED_POWER_KW, PERFORMANCE_RATIO

plt.style.use("ggplot")

ACTION_LABELS = {
    0: "IDLE",
    1: "CHARGE",
    2: "DISCHARGE",
}


## Build the environment

The next cell instantiates the environment and prints a compact summary so students can immediately see what their design choices imply.

In [ ]:
env = MicrogridLawEnv(
    reward_weights=reward_weights,
    selected_features=selected_features,
)

env_summary = {
    "selected_features": selected_features,
    "observation_space_shape": env.observation_space.shape,
    "action_space_n": env.action_space.n,
    "battery_capacity": env.max_soc,
    "max_charge_discharge_rate": env.max_rate,
    "initial_soc": 20.0,
    "episode_horizon_hours": 168,
    "daily_export_cap_fraction": 0.20,
    "solar_peak_kw": PANEL_COUNT * PANEL_RATED_POWER_KW * PERFORMANCE_RATIO,
    "reward_weights": reward_weights,
}

for key, value in env_summary.items():
    print(f"{key}: {value}")


## Training

We train a PPO agent on the environment configured above.

In [ ]:
model = PPO(
    "MlpPolicy",
    env,
    learning_rate=learning_rate,
    ent_coef=entropy_coef,
    gamma=gamma,
    verbose=0,
    seed=seed,
)

model.learn(total_timesteps=total_timesteps)


## Evaluation with an IDLE baseline

The RL agent is evaluated against a simple baseline that always stays idle.
This makes the profit improvement easier to interpret than looking at the PPO reward alone.

In [ ]:
def rollout_policy(environment, policy_fn, horizon=168, seed=42):
    obs, _ = environment.reset(seed=seed)
    history = {
        "rewards": [],
        "profits": [],
        "socs": [],
        "actions": [],
    }
    total_reward = 0.0
    total_profit = 0.0

    for _ in range(horizon):
        action = int(policy_fn(obs))
        obs, reward, terminated, truncated, info = environment.step(action)

        history["rewards"].append(float(reward))
        history["profits"].append(float(info["profit"]))
        history["socs"].append(float(info["soc"]))
        history["actions"].append(action)

        total_reward += float(reward)
        total_profit += float(info["profit"])

        if terminated or truncated:
            break

    return {
        "history": history,
        "total_reward": total_reward,
        "total_profit": total_profit,
        "steps": len(history["actions"]),
    }


def ppo_policy(observation):
    action, _ = model.predict(observation, deterministic=True)
    return int(action)


def idle_policy(_observation):
    return 0


rl_env = MicrogridLawEnv(reward_weights=reward_weights, selected_features=selected_features)
idle_env = MicrogridLawEnv(reward_weights=reward_weights, selected_features=selected_features)

rl_results = rollout_policy(rl_env, ppo_policy, horizon=168, seed=seed)
idle_results = rollout_policy(idle_env, idle_policy, horizon=168, seed=seed)

profit_delta = rl_results["total_profit"] - idle_results["total_profit"]

if abs(idle_results["total_profit"]) < 1e-9:
    profit_margin_vs_idle_pct = np.nan
else:
    profit_margin_vs_idle_pct = 100.0 * profit_delta / abs(idle_results["total_profit"])

print(f"PPO total reward: {rl_results['total_reward']:.2f}")
print(f"PPO total profit: {rl_results['total_profit']:.2f} MAD")
print(f"IDLE total profit: {idle_results['total_profit']:.2f} MAD")
print(f"Profit delta vs IDLE: {profit_delta:.2f} MAD")

if np.isnan(profit_margin_vs_idle_pct):
    print("Profit margin vs IDLE: undefined because the baseline profit is zero.")
else:
    print(f"Profit margin vs IDLE: {profit_margin_vs_idle_pct:.2f}%")


In [ ]:
fig, axs = plt.subplots(4, 1, figsize=(11, 13), sharex=True)

axs[0].plot(rl_results["history"]["rewards"])
axs[0].set_title("PPO reward over time")
axs[0].set_ylabel("Reward")

axs[1].plot(rl_results["history"]["socs"])
axs[1].set_title("Battery state of charge")
axs[1].set_ylabel("SOC")

axs[2].step(range(len(rl_results["history"]["actions"])), rl_results["history"]["actions"], where="mid", label="PPO")
axs[2].step(range(len(idle_results["history"]["actions"])), idle_results["history"]["actions"], where="mid", linestyle="--", label="IDLE")
axs[2].set_title("Actions: PPO vs IDLE baseline")
axs[2].set_ylabel("Action")
axs[2].legend()

axs[3].plot(np.cumsum(rl_results["history"]["profits"]), label="PPO cumulative profit")
axs[3].plot(np.cumsum(idle_results["history"]["profits"]), linestyle="--", label="IDLE cumulative profit")
axs[3].set_title("Cumulative profit comparison")
axs[3].set_ylabel("MAD")
axs[3].set_xlabel("Hour")
axs[3].legend()

plt.tight_layout()
plt.show()


## Save a submission bundle for instructor evaluation

The next cell exports a single zip archive containing:
- the trained Stable-Baselines3 model,
- a metadata file with observation-space shape, selected features, reward weights, hyperparameters, package versions, and evaluation results,
- a binary feature mask for compatibility with simple instructor tooling,
- the environment source files used by the notebook.

This gives the instructor enough information to reload the model and rebuild the exact environment setup.

In [ ]:
models_dir = Path("models")
models_dir.mkdir(exist_ok=True)

model_stem = f"{student_id}_model"
model_path_without_suffix = models_dir / model_stem
model.save(model_path_without_suffix.as_posix())
saved_model_zip = models_dir / f"{model_stem}.zip"

# Convert NumPy scalar/array types to Python-native objects for JSON serialization.
def to_json_safe(obj):
    if isinstance(obj, np.generic):
        return obj.item()
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    raise TypeError(f"Object of type {type(obj).__name__} is not JSON serializable")

feature_mask = [1 if feature in selected_features else 0 for feature in env.feature_names]

metadata = {
    "student_id": student_id,
    "selected_features": selected_features,
    "feature_order_reference": env.feature_names,
    "feature_mask": feature_mask,
    "observation_space_shape": list(env.observation_space.shape),
    "action_space_n": env.action_space.n,
    "action_labels": ACTION_LABELS,
    "reward_weights": reward_weights,
    "hyperparameters": {
        "algorithm": "PPO",
        "total_timesteps": total_timesteps,
        "learning_rate": learning_rate,
        "gamma": gamma,
        "seed": seed,
    },
    "environment": {
        "episode_horizon_hours": 168,
        "battery_capacity": env.max_soc,
        "max_charge_discharge_rate": env.max_rate,
        "initial_soc": 20.0,
        "daily_export_cap_fraction": 0.20,
        "solar_panel_count": PANEL_COUNT,
        "panel_rated_power_kw": PANEL_RATED_POWER_KW,
        "performance_ratio": PERFORMANCE_RATIO,
    },
    "evaluation": {
        "ppo_total_reward": rl_results["total_reward"],
        "ppo_total_profit_mad": rl_results["total_profit"],
        "idle_total_profit_mad": idle_results["total_profit"],
        "profit_delta_vs_idle_mad": profit_delta,
        "profit_margin_vs_idle_pct": None if np.isnan(profit_margin_vs_idle_pct) else float(profit_margin_vs_idle_pct),
    },
    "package_versions": {
        "python": "3.11",
        "numpy": np.__version__,
        "gymnasium": gymnasium.__version__,
        "stable_baselines3": stable_baselines3.__version__,
    },
}

metadata_path = models_dir / f"{student_id}_metadata.json"
mask_path = models_dir / f"{student_id}_mask.json"
bundle_path = models_dir / f"{student_id}_submission_bundle.zip"

with metadata_path.open("w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, default=to_json_safe)

with mask_path.open("w", encoding="utf-8") as f:
    json.dump({"mask": feature_mask, "selected_features": selected_features}, f, indent=2, default=to_json_safe)

with zipfile.ZipFile(bundle_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(saved_model_zip, arcname=saved_model_zip.name)
    archive.write(metadata_path, arcname=metadata_path.name)
    archive.write(mask_path, arcname=mask_path.name)
    archive.write(Path("env/microgrid_env.py"), arcname="env/microgrid_env.py")
    archive.write(Path("env/models.py"), arcname="env/models.py")

print(f"Saved model zip: {saved_model_zip}")
print(f"Saved metadata: {metadata_path}")
print(f"Saved bundle: {bundle_path}")


## Example: how an instructor can reload the bundle

This final cell demonstrates that the exported zip contains enough information to recreate the environment and load the policy.

In [ ]:
def load_submission_bundle(bundle_file):
    bundle_file = Path(bundle_file)
    extract_dir = bundle_file.with_suffix("")
    extract_dir.mkdir(exist_ok=True)

    with zipfile.ZipFile(bundle_file, "r") as archive:
        archive.extractall(extract_dir)

    metadata_file = next(extract_dir.glob("*_metadata.json"))
    with metadata_file.open("r", encoding="utf-8") as f:
        loaded_metadata = json.load(f)

    loaded_model = PPO.load(extract_dir / f"{loaded_metadata['student_id']}_model.zip")
    loaded_env = MicrogridLawEnv(
        reward_weights=loaded_metadata["reward_weights"],
        selected_features=loaded_metadata["selected_features"],
    )
    return loaded_model, loaded_env, loaded_metadata


loaded_model, loaded_env, loaded_metadata = load_submission_bundle(bundle_path)
print("Reloaded student:", loaded_metadata["student_id"])
print("Reloaded observation shape:", loaded_env.observation_space.shape)
print("Reloaded selected features:", loaded_metadata["selected_features"])
